In [61]:
import torch
from transformers import AutoTokenizer, AutoModel

model_name = "Jihuai/bert-ancient-chinese"
model_name="Jihuai/bert-ancient-chinese"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, output_attentions=True)

# char_1, char_2: linear order
# idx_a, idx_b: directionality of attention
def get_character_attention(text, char_1, char_2, dir="forward"):    
    inputs = tokenizer(text, return_tensors="pt")
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    
    print(f"Tokenized Sequence: {tokens}\n")
    
    if dir == "forward":
        try:
            idx_a = tokens.index(char_1)
            idx_b = next(i for i in reversed(range(len(tokens))) if tokens[i] == char_2)
        except ValueError as e:
            raise ValueError(f"Ensure both '{char_1}' and '{char_2}' exist as distinct tokens in: {tokens}") from e
    else:
        try:
            idx_b = tokens.index(char_1)
            idx_a = next(i for i in reversed(range(len(tokens))) if tokens[i] == char_2)
        except ValueError as e:
            raise ValueError(f"Ensure both '{char_1}' and '{char_2}' exist as distinct tokens in: {tokens}") from e

    with torch.no_grad():
        outputs = model(**inputs)
    
    attentions = outputs.attentions 
    last_hidden_state = outputs.last_hidden_state[0]
    embedding_a = last_hidden_state[idx_a]
    embedding_b = last_hidden_state[idx_b]
    
    num_layers = len(attentions)
    num_heads = attentions[0].shape[1]
    
    print(f"--- Attention from '{char_1}' (Token {idx_a}) targeting '{char_2}' (Token {idx_b}) ---")
    
    for layer in range(num_layers):
        # print(f"Layer {layer + 1}:")
        layer_attention = attentions[layer][0] 
        
        for head in range(num_heads):
            attn_val = layer_attention[head, idx_a, idx_b].item()
            # print(f"  Head {head + 1}: {attn_val:.4f}")
            
    return (attentions, embedding_a, embedding_b, idx_a, idx_b)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 31477.25it/s]
BertModel LOAD REPORT from: Jihuai/bert-ancient-chinese
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialize

In [62]:
sentence = "其殉國死義乎"
character_1 = "其"
character_2 = "乎"

attentions, embed_a, embed_b, pos_a, pos_b = get_character_attention(
    text=sentence, 
    char_1=character_1, 
    char_2=character_2)

Tokenized Sequence: ['[CLS]', '其', '殉', '國', '死', '義', '乎', '[SEP]']

--- Attention from '其' (Token 1) targeting '乎' (Token 6) ---


In [64]:
attentions, embed_a, embed_b, pos_a, pos_b = get_character_attention(
    text=sentence, 
    char_1=character_1, 
    char_2=character_2, 
    dir="backward"
)

Tokenized Sequence: ['[CLS]', '其', '殉', '國', '死', '義', '乎', '[SEP]']

--- Attention from '其' (Token 6) targeting '乎' (Token 1) ---


In [44]:
import pandas as pd
with open(r"..\output\01_extract_pairs\combosDF.csv", "r", encoding="utf-8") as in_file:
    combo_df = pd.read_csv(in_file)

In [45]:
for item in combo_df.iterrows():
    print(item)
    break

(0, sfp                           NaN
dc                            NaN
docID                KR1d0052_001
paragraphID                     0
sentenceID     KR1d0052_001_title
sentence                      曲禮上
Name: 0, dtype: object)


In [53]:
attention_list = list()

In [ ]:
for (i, item) in combo_df.iterrows():
    # NaN checks
    if item["dc"] != item["dc"]:
        continue
    if item["sfp"] != item["sfp"]:
        continue

    sentence = item["sentence"]
    character_1 = item["dc"]
    character_2 = item["sfp"]

    try:
        (attentions_for, pos_a, pos_b) = get_character_attention(
            text=sentence, 
            char_1=character_1, 
            char_2=character_2, 
            dir="forward"
        )
        item["attentions_for"] = attentions_for
        item["pos_dc"] = pos_a
        item["pos_sfp"] = pos_b
    except Exception as e:
        item["attentions_for"] = None
        item["pos_dc"] = None
        item["pos_sfp"] = None
    
    try:
        (attentions_back, _, _) = get_character_attention(
            text=sentence, 
            char_1=character_1, 
            char_2=character_2, 
            dir="backward"
        )
        item["attentions_back"] = attentions_back
    except Exception as e:
        item["attentions_back"] = None

        
    attention_list.append(item)

Tokenized Sequence: ['[CLS]', '不', '亦', '禽', '獸', '之', '心', '乎', '[SEP]']

--- Attention from '不' (Token 1) targeting '乎' (Token 7) ---
Tokenized Sequence: ['[CLS]', '不', '亦', '禽', '獸', '之', '心', '乎', '[SEP]']

--- Attention from '不' (Token 7) targeting '乎' (Token 1) ---
Tokenized Sequence: ['[CLS]', '非', '禮', '也', '[SEP]']

--- Attention from '非' (Token 1) targeting '也' (Token 3) ---
Tokenized Sequence: ['[CLS]', '非', '禮', '也', '[SEP]']

--- Attention from '非' (Token 3) targeting '也' (Token 1) ---
Tokenized Sequence: ['[CLS]', '亦', '非', '禮', '也', '[SEP]']

--- Attention from '亦' (Token 1) targeting '也' (Token 4) ---
Tokenized Sequence: ['[CLS]', '亦', '非', '禮', '也', '[SEP]']

--- Attention from '亦' (Token 4) targeting '也' (Token 1) ---
Tokenized Sequence: ['[CLS]', '亦', '非', '禮', '也', '[SEP]']

--- Attention from '非' (Token 2) targeting '也' (Token 4) ---
Tokenized Sequence: ['[CLS]', '亦', '非', '禮', '也', '[SEP]']

--- Attention from '非' (Token 4) targeting '也' (Token 2) ---
Tokenized Se

In [59]:
attention_list[:5]

[sfp                                                                乎
 dc                                                                 不
 docID                                                   KR1d0052_001
 paragraphID                                                        6
 sentenceID                                 KR1d0052_001_par6_110-116
 sentence                                                     不亦禽獸之心乎
 attentions_for     ([[tensor([[0.0090, 0.4664, 0.0919, 0.0458, 0....
 pos_dc                                                             1
 pos_sfp                                                            7
 attentions_back    ([[tensor([[0.0090, 0.4664, 0.0919, 0.0458, 0....
 Name: 46, dtype: object,
 sfp                                                                也
 dc                                                                 非
 docID                                                   KR1d0052_001
 paragraphID                                                    

In [60]:
import pickle as pkl
with open(r"..\output\04_bert\attention_list_ancient.pkl", "w", encoding="utf-8") as out_file:
    pkl.dump(attention_list, out_file)

TypeError: write() argument must be str, not bytes

In [3]:
import matplotlib.pyplot as plt

# Plot just the first attention map in the 'attentions' tuple for simplicity
layer_attention = attentions[0]
attention_map_1st_head_to_a_at_b = layer_attention[0, pos_a, pos_b]

plt.imshow(attention_map_1st_head_to_a_at_b, cmap='hot', interpolation='nearest')
plt.title(f"Attention from {character_1} to {character_2} - 1st Head")
# Optional: plt.colorbar()
plt.show()

NameError: name 'attentions' is not defined

In [1]:
import pandas as pd
    attentions_df = pd.DataFrame(attentions, columns=[f'Head_{i}' for i in range(13)])
    print(attentions_df.describe())

IndentationError: unexpected indent (1685946050.py, line 2)

In [65]:
print(f"Shape of embed_a: {embed_a.shape}")
print(f"Shape of embed_b: {embed_b.shape}")

Shape of embed_a: torch.Size([768])
Shape of embed_b: torch.Size([768])
